In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path.home() / "Downloads" / "transformer_data"

train_df = pd.read_csv(DATA_DIR / "Train.csv")
val_df = pd.read_csv(DATA_DIR / "Validation.csv")
test_df = pd.read_csv(DATA_DIR / "Test.csv")

print(f"Train: {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test: {test_df.shape}")

train_df.head()

Train: (50420, 6)
Validation: (7047, 6)
Test: (6981, 6)


,video_id,label,frames,label_id,shape,format
0,1,Doing other things,37,0,"(100, 176)",JPEG
1,3,Pushing Two Fingers Away,37,6,"(100, 176)",JPEG
2,6,Drumming Fingers,37,1,"(100, 176)",JPEG
3,11,Sliding Two Fingers Down,37,10,"(100, 176)",JPEG
4,14,Pushing Hand Away,37,5,"(100, 176)",JPEG


In [2]:
labels = (
    pd.concat([train_df["label"], val_df["label"]])
    .dropna()
    .unique()
)

for label in sorted(labels):
    print(label)

print(f"\nTotal distinct labels: {len(labels)}")

Doing other things
Drumming Fingers
No gesture
Pulling Hand In
Pulling Two Fingers In
Pushing Hand Away
Pushing Two Fingers Away
Rolling Hand Backward
Rolling Hand Forward
Shaking Hand
Sliding Two Fingers Down
Sliding Two Fingers Left
Sliding Two Fingers Right
Sliding Two Fingers Up
Stop Sign
Swiping Down
Swiping Left
Swiping Right
Swiping Up
Thumb Down
Thumb Up
Turning Hand Clockwise
Turning Hand Counterclockwise
Zooming In With Full Hand
Zooming In With Two Fingers
Zooming Out With Full Hand
Zooming Out With Two Fingers

Total distinct labels: 27


In [5]:
TARGET_LABELS = [
    "Doing other things",
    "No gesture",
    "Swiping Down",
    "Swiping Left",
    "Swiping Right",
    "Swiping Up",
]

train = train_df.copy()
val = val_df.copy()
test = test_df.rename(columns={"id": "video_id"}).copy()

combined_df = pd.concat([train, val, test], ignore_index=True)
filtered_df = combined_df[combined_df["label"].isin(TARGET_LABELS)].copy()

# Merge "Doing other things" into "No gesture"
mask = filtered_df["label"] == "Doing other things"
filtered_df.loc[mask, ["label", "label_id"]] = ["No gesture", 2]

OUTPUT_PATH = Path("updated_data.csv")
filtered_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(filtered_df)} rows to {OUTPUT_PATH.resolve()}")
print(filtered_df["label"].value_counts())

Saved 15238 rows to /Users/aryamanwade/Desktop/ml_projects/remote_desktop/remote-desktop/updated_data.csv
label
No gesture       7187
Swiping Down     2072
Swiping Left     2009
Swiping Up       2009
Swiping Right    1961
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

train_split, temp_split = train_test_split(
    filtered_df,
    test_size=0.2,
    stratify=filtered_df["label"],
    random_state=42,
)

val_split, test_split = train_test_split(
    temp_split,
    test_size=0.5,
    stratify=temp_split["label"],
    random_state=42,
)

train_split.to_csv("train.csv", index=False)
val_split.to_csv("val.csv", index=False)
test_split.to_csv("test.csv", index=False)

print(f"Train: {len(train_split)} ({len(train_split) / len(filtered_df):.1%})")
print(f"Val:   {len(val_split)} ({len(val_split) / len(filtered_df):.1%})")
print(f"Test:  {len(test_split)} ({len(test_split) / len(filtered_df):.1%})")

print("\nTrain label distribution:")
print(train_split["label"].value_counts(normalize=True).round(3))
print("\nVal label distribution:")
print(val_split["label"].value_counts(normalize=True).round(3))
print("\nTest label distribution:")
print(test_split["label"].value_counts(normalize=True).round(3))

Train: 12190 (80.0%)
Val:   1524 (10.0%)
Test:  1524 (10.0%)

Train label distribution:
label
No gesture       0.472
Swiping Down     0.136
Swiping Left     0.132
Swiping Up       0.132
Swiping Right    0.129
Name: proportion, dtype: float64

Val label distribution:
label
No gesture       0.472
Swiping Down     0.136
Swiping Left     0.132
Swiping Up       0.132
Swiping Right    0.129
Name: proportion, dtype: float64

Test label distribution:
label
No gesture       0.472
Swiping Down     0.136
Swiping Left     0.132
Swiping Up       0.132
Swiping Right    0.129
Name: proportion, dtype: float64
